# Modele de detection de plaques — Ciment's EyeCe carnet entraine le modele `plate`, qui **localise** la plaque sur l'imaged'un vehicule. Il ne la lit pas : la lecture est faite ensuite par easyocr, surla zone que ce modele decoupe.## Pourquoi un modele dedieSans lui, la localisation se fait par traitement d'image classique (chapeauhaut-de-forme noir + Sobel). Cela repere des rectangles contrastes — dontbeaucoup ne sont pas des plaques : un autocollant, une calandre, un reflet.Chaque faux candidat coute une seconde d'OCR sur une machine a deux coeurs.Un modele YOLO dedie change deux choses : il ne propose que des plaques, et illes cadre serre, ce qui augmente nettement le taux de lecture.## Ce qui limite vraiment la lectureAvant d'entrainer quoi que ce soit, verifiez le cadrage. Mesure faite sur lavideo d'essai du projet : la plaque du vehicule detecte faisait **60 pixels delarge**. Aucun moteur ne lit cela. Il en faut **au moins 90, en pratique 120**.Aucun modele ne rattrapera une camera trop loin. Regardez d'abord`Parametres -> Etat du systeme -> Lecture des plaques` : la colonne « largeurvue » vous donne le chiffre reel de votre installation.

## 1. EnvironnementSur Colab : `Execution -> Modifier le type d'execution -> GPU T4`.

In [ ]:
!nvidia-smi!pip install -q ultralytics roboflow

## 2. Le jeu de donnees`license-plate-recognition-rxg4e` (Roboflow Universe) : environ 24 000 imagesde vehicules avec la plaque annotee, prises de face et de trois quarts, de jourcomme de nuit. C'est le jeu le plus utilise pour cette tache.Collez votre cle Roboflow (Settings -> API Keys).

In [ ]:
from roboflow import Roboflowrf = Roboflow(api_key="VOTRE_CLE_ICI")projet = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")jeu = projet.version(4).download("yolov8")print(jeu.location)

## 3. EntrainementUne seule classe, objet petit et tres regulier : le modele converge vite.`imgsz=640` est **obligatoire** — c'est la taille figee a l'export OpenVINO duprojet, et la changer casse l'inference.

In [ ]:
from ultralytics import YOLOmodele = YOLO("yolov8n.pt")modele.train(    data=f"{jeu.location}/data.yaml",    epochs=40,    imgsz=640,    batch=32,    patience=10,    name="plaques",)

## 4. Verifier avant de livrerRegardez le mAP50 de la classe plaque. En dessous de 0,85, le modele cadreramal et la lecture en patira. Testez aussi sur une image de VOTRE portail.

In [ ]:
metriques = modele.val()print("mAP50 :", metriques.box.map50)# Essai sur une image reelle du site — deposez-la dans Colab d'abord.# resultats = modele.predict("portail.jpg", conf=0.35, save=True)

## 5. Recuperer le modeleTelechargez `best.pt` et deposez-le dans `models/` sous le nom`ciments_eye_plate_best.pt`, puis exportez-le en OpenVINO :```python scripts/export_openvino.py```Ajoutez enfin cette entree dans `config/config.yaml`, section `models` :```yaml  plate:    file: models/ciments_eye_plate_best_openvino_model    conf: 0.35    enabled: true```Le lecteur de plaques s'en sert automatiquement des qu'il est declare.

In [ ]:
from google.colab import filesfiles.download("runs/detect/plaques/weights/best.pt")